# Vision Data Pipelines Inspection & Augmentation Visualizer

This notebook demonstrates how to load, inspect, and visualize the three datasets supported in this repo:
1. **CIFAR-100** (32x32, 100 classes)
2. **Tiny-ImageNet-200** (64x64, 200 classes)
3. **ImageNet Subset (ImageNet-100 / Mock ImageNet)** (224x224, 100 classes)
4. **Mixup and CutMix** data augmentation visualizer

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure project root is in sys.path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils.seed import set_seed
set_seed(42)
print("Repo root:", REPO_ROOT)

## 1. CIFAR-100 Exploration

In [ ]:
from src.data.cifar100 import get_cifar100_datasets
from src.data.transforms import DATASET_STATS

train_ds, val_ds, test_ds = get_cifar100_datasets(data_dir=REPO_ROOT / "data/cifar100", val_split=0.1)
print(f"CIFAR-100 Train size: {len(train_ds)}")
print(f"CIFAR-100 Val size:   {len(val_ds)}")
print(f"CIFAR-100 Test size:  {len(test_ds)}")

# Denormalize helper for plotting
def unnormalize(tensor, stats_key="cifar100"):
    mean = np.array(DATASET_STATS[stats_key]["mean"])
    std = np.array(DATASET_STATS[stats_key]["std"])
    img = tensor.permute(1, 2, 0).cpu().numpy()
    img = std * img + mean
    return np.clip(img, 0, 1)

# Display sample grid
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flat):
    img_t, target = train_ds[i]
    ax.imshow(unnormalize(img_t, "cifar100"))
    ax.set_title(f"Class {target}")
    ax.axis("off")
plt.suptitle("CIFAR-100 Samples with Training Augmentations", fontsize=14)
plt.tight_layout()
plt.show()

## 2. Tiny-ImageNet-200 Exploration

In [ ]:
from src.data.tiny_imagenet import get_tiny_imagenet_datasets

# Set download=True if you want to automatically download the 248MB dataset from Stanford
try:
    tiny_train, tiny_val, _ = get_tiny_imagenet_datasets(data_dir=REPO_ROOT / "data/tiny_imagenet", download=False)
    print(f"Tiny-ImageNet Train samples: {len(tiny_train)}")
    print(f"Tiny-ImageNet Val samples:   {len(tiny_val)}")
    print(f"Classes: {len(tiny_train.classes)}")
except Exception as e:
    print("Tiny-ImageNet not yet downloaded locally. Use `python scripts/download_data.py --dataset tiny_imagenet` to download.")

## 3. ImageNet Subset / Mock Dataset Exploration

In [ ]:
from src.data.imagenet_subset import get_imagenet_subset_datasets

# If full ImageNet is not present, allow_mock=True automatically generates a lightweight mock dataset
imgnet_train, imgnet_val, _ = get_imagenet_subset_datasets(
    data_dir=REPO_ROOT / "data/imagenet",
    num_classes=10,
    image_size=224,
    allow_mock=True,
)

print(f"ImageNet Subset Train samples: {len(imgnet_train)}")
print(f"ImageNet Subset Val samples:   {len(imgnet_val)}")
print(f"Selected Classes: {imgnet_train.classes}")

img_t, target = imgnet_train[0]
print(f"Image tensor shape: {img_t.shape}, target label: {target}")

## 4. Visualizing Mixup & CutMix Augmentation
Mixup and CutMix blend pairs of images and interpolate their label distributions, which stabilizes training for vision MLPs.

In [ ]:
from src.data.transforms import MixupCutmixCollate

collate = MixupCutmixCollate(num_classes=100, mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0)

# Take 4 samples from CIFAR-100
raw_batch = [train_ds[i] for i in range(4)]
mixed_images, mixed_targets = collate(raw_batch)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i, ax in enumerate(axes):
    ax.imshow(unnormalize(mixed_images[i], "cifar100"))
    top_classes = torch.topk(mixed_targets[i], k=2)
    title = f"c{top_classes.indices[0].item()}: {top_classes.values[0].item():.2f}\n" \
            f"c{top_classes.indices[1].item()}: {top_classes.values[1].item():.2f}"
    ax.set_title(title, fontsize=10)
    ax.axis("off")
plt.suptitle("Mixup / CutMix Augmented Batch & Mixed Soft Targets", fontsize=13)
plt.tight_layout()
plt.show()